In [1]:
import pandas as pd

# Load only the first 5 rows to inspect the structure
print("--- COLUMN HEADERS ---")

df_patients = pd.read_csv("patients.csv", nrows=5)
print(f"Patients: {df_patients.columns.tolist()}")

df_diagnoses = pd.read_csv("diagnoses.csv", nrows=5)
print(f"Diagnoses: {df_diagnoses.columns.tolist()}")

df_meds = pd.read_csv("medications.csv", nrows=5)
print(f"Medications: {df_meds.columns.tolist()}")

df_labs = pd.read_csv("lab_results.csv", nrows=5)
print(f"Labs: {df_labs.columns.tolist()}")

df_outcomes = pd.read_csv("outcomes.csv", nrows=5)
print(f"Outcomes: {df_outcomes.columns.tolist()}")

--- COLUMN HEADERS ---
Patients: ['patient_id', 'age', 'sex', 'bmi', 'systolic_bp', 'diastolic_bp', 'heart_rate', 'temperature_f', 'smoking_status', 'alcohol_use', 'exercise_level', 'insurance_type', 'charlson_index', 'dx_hypertension', 'dx_type2_diabetes', 'dx_hyperlipidemia', 'dx_obesity', 'dx_coronary_artery_disease', 'dx_heart_failure', 'dx_atrial_fibrillation', 'dx_chronic_kidney_disease', 'dx_copd', 'dx_asthma', 'dx_depression', 'dx_anxiety', 'dx_hypothyroidism', 'dx_osteoarthritis', 'dx_type1_diabetes']
Diagnoses: ['patient_id', 'visit_date', 'visit_type', 'primary_diagnosis', 'primary_icd10', 'secondary_diagnoses', 'secondary_icd10s', 'provider_specialty']
Medications: ['patient_id', 'medication', 'dose', 'unit', 'frequency', 'indication', 'start_date', 'duration_days', 'is_generic', 'adherence_pct']
Labs: ['patient_id', 'test_date', 'test_name', 'value', 'unit', 'reference_low', 'reference_high', 'flag', 'is_abnormal', 'delta_from_normal']
Outcomes: ['patient_id', 'admission_d

In [1]:
import pandas as pd

print("Aggregating Medications...")
df_meds = pd.read_csv("medications.csv")
meds_agg = df_meds.groupby('patient_id').agg(
    avg_adherence=('adherence_pct', 'mean'),
    total_meds=('medication', 'count')
).reset_index()

print("Aggregating Labs...")
df_labs = pd.read_csv("lab_results.csv")
labs_agg = df_labs.groupby('patient_id').agg(
    abnormal_labs=('is_abnormal', lambda x: (x == True).sum()),
    total_tests=('test_name', 'count')
).reset_index()

print("Aggregating Outcomes...")
df_outcomes = pd.read_csv("outcomes.csv")
outcomes_agg = df_outcomes.groupby('patient_id').agg(
    total_admissions=('admission_date', 'count'),
    icu_admissions=('icu_admission', lambda x: (x == True).sum())
).reset_index()

print("Loading Base Patient Table...")
master_df = pd.read_csv("patients.csv")

print("Executing Smart Merge...")
master_df = master_df.merge(meds_agg, on='patient_id', how='left')
master_df = master_df.merge(labs_agg, on='patient_id', how='left')
master_df = master_df.merge(outcomes_agg, on='patient_id', how='left')

# Fill missing values with 0 (e.g., if a patient had no lab tests)
master_df.fillna(0, inplace=True)

master_df.to_csv("final_clean_master.csv", index=False)
print(f"Success. Compressed dataset has {len(master_df)} rows.")

Aggregating Medications...
Aggregating Labs...
Aggregating Outcomes...
Loading Base Patient Table...
Executing Smart Merge...
Success. Compressed dataset has 100000 rows.
